### Expert Knowledge Worker

In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [2]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [5]:
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/netflix-dataset.csv"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 1 files in the knowledge base
Total characters in knowledge base: 3,390,899


In [6]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 948,063


In [41]:
from langchain_community.document_loaders import CSVLoader

# Define the path to your specific CSV file
file_path = "knowledge-base/netflix-dataset.csv"

# Initialize the loader
# Use 'metadata_columns' to automatically move the 'type' column into the metadata dict
loader = CSVLoader(
    file_path=file_path,
    encoding='utf-8',
    metadata_columns=["type"]  # This extracts 'Movie' or 'TV Show' into metadata
)

# Load the data into document objects
documents = loader.load()

# Update the metadata key name to 'doc_type' as per your requirement
for doc in documents:
    # 'type' is now in metadata because of the loader configuration above
    doc.metadata["show_type"] = doc.metadata.get("type")
    
    # Optional: Remove the original 'type' key if you only want 'doc_type'
    # del doc.metadata["type"]

print(f"Loaded {len(documents)} rows.")
print(f"Example Metadata: {documents[0].metadata}")

Loaded 8807 rows.
Example Metadata: {'source': 'knowledge-base/netflix-dataset.csv', 'row': 0, 'type': 'Movie', 'show_type': 'Movie'}


In [42]:
documents[1]

Document(metadata={'source': 'knowledge-base/netflix-dataset.csv', 'row': 1, 'type': 'TV Show', 'show_type': 'TV Show'}, page_content='show_id: s2\ntitle: Blood & Water\ndirector: \ncast: Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, Dillon Windvogel, Natasha Thahane, Arno Greeff, Xolile Tshabalala, Getmore Sithole, Cindy Mahlangu, Ryle De Morny, Greteli Fincham, Sello Maake Ka-Ncube, Odwa Gwanya, Mekaila Mathys, Sandi Schultz, Duane Williams, Shamilla Miller, Patrick Mofokeng\ncountry: South Africa\ndate_added: September 24, 2021\nrelease_year: 2021\nrating: TV-MA\nduration: 2 Seasons\nlisted_in: International TV Shows, TV Dramas, TV Mysteries\ndescription: After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at birth.')

In [43]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 8818 chunks
First chunk:

page_content='show_id: s1
title: Dick Johnson Is Dead
director: Kirsten Johnson
cast: 
country: United States
date_added: September 25, 2021
release_year: 2020
rating: PG-13
duration: 90 min
listed_in: Documentaries
description: As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.' metadata={'source': 'knowledge-base/netflix-dataset.csv', 'row': 0, 'type': 'Movie', 'show_type': 'Movie'}


In [44]:
chunks[100]

Document(metadata={'source': 'knowledge-base/netflix-dataset.csv', 'row': 100, 'type': 'TV Show', 'show_type': 'TV Show'}, page_content="show_id: s101\ntitle: Tobot Galaxy Detectives\ndirector: \ncast: Austin Abell, Travis Turner, Cole Howard, Anna Cummer, Jesse Inocalla, Brian Dobson, Michael Adamthwaite, Joseph Girgis, Caitlyn Bairstow\ncountry: \ndate_added: September 7, 2021\nrelease_year: 2019\nrating: TV-Y7\nduration: 2 Seasons\nlisted_in: Kids' TV\ndescription: An intergalactic device transforms toy cars into robots: the Tobots! Working with friends to solve mysteries, they protect the world from evil.")

### Make vectors and store in Chroma

In [45]:
# Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorstore created with 8818 documents


In [46]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 8,818 vectors with 384 dimensions in the vector store


In [55]:
 # Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])

documents = result['documents']
metadatas = result['metadatas']

show_type = [metadata['show_type'] for metadata in metadatas]
shows = [['Red', 'Green'][['Movie','TV Show'].index(t)] for t in show_type]


In [57]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=shows, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(show_type, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [58]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=shows, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(show_type, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()